# 12 Predictor Agreement


## Method

Goal: compare whether trained disorder predictors agree more strongly with each other than the experimental annotation sources behind the corresponding UdonPred dataset-specific models.

This is not an accuracy benchmark against a single ground truth. It is a diagnostic comparison between two agreement levels:

1. Predictor agreement on the human proteome
2. Annotation agreement on overlapping experimentally annotated proteins/residues

The predictor comparison uses matched residues from the same human proteome proteins. For every predictor pair, the script reports residue-level Spearman/Pearson correlation, protein-mean Spearman/Pearson correlation, and z-score MAE. CheZOD, pLDDT, and ADOPT are negated before comparison because their native direction is lower = more disorder.

The annotation comparison joins each UdonPred predictor pair to the matching annotation-ceiling pair. Annotation values come from the original experimental label datasets and are computed only on proteins/residues that overlap between those datasets, not on the human proteome predictions.

The main diagnostic is the difference between predictor Spearman and annotation Spearman. Positive values mean the trained predictors agree more strongly on the human proteome than the underlying experimental annotations agree on their overlapping labeled data. This should be interpreted as model-output convergence, not as proof that the models are better than the annotations.

In [ ]:
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RESULTS = ROOT / "results"
OUTPUT_DIR = RESULTS / "compare_predictors_with_all_predictors_wo_pdbflex"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pairwise_csv = OUTPUT_DIR / "pairwise_agreement.csv"
contested_csv = OUTPUT_DIR / "contested_regions.csv"

predictor_args = [
    "trizod=results/human_proteome/UdonPred/trizod",
    "chezod=results/human_proteome/UdonPred/chezod",
    "softdis=results/human_proteome/UdonPred/softdis",
    "atlas=results/human_proteome/UdonPred/atlas",
    "plddt=results/human_proteome/UdonPred/plddt",
    "disprot=results/human_proteome/UdonPred/disprot",
    "SETH=results/human_proteome/SETH/seth_human_proteome.caid",
    "IUPred3=results/human_proteome/IUPred3",
    "ADOPT=results/human_proteome/ADOPT",
    "metapredict=results/human_proteome/metapredict/metapredict_human_proteome.caid",
    "PUNCH2_light=results/human_proteome/punch2_light/disorder",
    "DisoFLAG=results/human_proteome/DisoFLAG/caid",
    "DisorderUnetLM=results/human_proteome/DisorderUnetLM/disorder",
    "DisPredict3=results/human_proteome/DisPredict3_native/caid",
]

required_pairwise_columns = {
    "residue_zscore_mae_std",
    "residue_zscore_mae_se",
    "protein_zscore_mae_std",
    "protein_zscore_mae_se",
}
needs_pairwise_update = (
    not pairwise_csv.exists()
    or not required_pairwise_columns.issubset(pd.read_csv(pairwise_csv, nrows=0).columns)
)

if not needs_pairwise_update and contested_csv.exists():
    print("Predictor comparison results already exist, skipping computation.")
else:
    print("Running predictor comparison...")
    cmd = [sys.executable, str(ROOT / "scripts" / "compare_predictors.py")]
    for value in predictor_args:
        cmd.extend(["--predictor", value])
    cmd.extend(["--output-dir", str(OUTPUT_DIR)])
    result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    result.check_returncode()

pairwise = pd.read_csv(pairwise_csv)
contested_regions = pd.read_csv(contested_csv)

pairwise.head()

## Predictor-Predictor Agreement

Residue-level correlation asks whether predictors rank individual residues similarly. Protein-level correlation asks whether they agree on which proteins are globally more disorder-prone.

In [ ]:
predictor_order = [
    name
    for name in ["trizod", "chezod", "softdis", "atlas", "plddt", "disprot", "SETH", "IUPred3", "ADOPT", "metapredict", "PUNCH2_light", "DisoFLAG", "DisorderUnetLM", "DisPredict3"]
    if name in set(pairwise["predictor_a"]) | set(pairwise["predictor_b"])
]

def pair_matrix(value_column):
    matrix = pd.DataFrame(np.nan, index=predictor_order, columns=predictor_order)
    for predictor in predictor_order:
        matrix.loc[predictor, predictor] = 1.0
    for row in pairwise.itertuples(index=False):
        matrix.loc[row.predictor_a, row.predictor_b] = getattr(row, value_column)
        matrix.loc[row.predictor_b, row.predictor_a] = getattr(row, value_column)
    return matrix

residue_spearman_matrix = pair_matrix("residue_spearman")
protein_spearman_matrix = pair_matrix("protein_spearman")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(
    residue_spearman_matrix,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    vmin=np.min([np.nanmin(residue_spearman_matrix.values),np.nanmin(protein_spearman_matrix.values)]),
    vmax=1,
    square=True,
    ax=axes[0],
)
axes[0].set_title("Residue-Level Spearman")
sns.heatmap(
    protein_spearman_matrix,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    vmin=np.min([np.nanmin(residue_spearman_matrix.values),np.nanmin(protein_spearman_matrix.values)]),
    vmax=1,
    square=True,
    ax=axes[1],
)
axes[1].set_title("Protein-Level Spearman")
# Rotate x tick labels so long labels are visible
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right')
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right')
fig.tight_layout()
# Increase margins to give room for rotated labels
fig.subplots_adjust(bottom=0.22, left=0.18)
fig.savefig(OUTPUT_DIR / "predictor_spearman_heatmaps.png", dpi=210, bbox_inches='tight', pad_inches=0.05)
plt.show()

pairwise.sort_values("residue_spearman", ascending=False)

### Why Is DisPredict3 Agreement Lower at Protein Level?

Protein-level Spearman correlation is calculated after averaging all residue scores within each protein. For a more direct comparison, the following analysis asks how many of the proteins placed in the top 10% by DisPredict3 are also placed in the top 10% by another predictor.

A value of 100% means that both predictors select exactly the same proteins. Approximately 10% overlap is expected for two unrelated top-10% selections.

In [ ]:
from IPython.display import display

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.compare_predictors import load_predictions

protein_mean_paths = {
    "DisPredict3": RESULTS / "human_proteome/DisPredict3_native/caid",
    "DisoFLAG": RESULTS / "human_proteome/DisoFLAG/caid",
    "DisorderUnetLM": RESULTS / "human_proteome/DisorderUnetLM/disorder",
    "PUNCH2_light": RESULTS / "human_proteome/punch2_light/disorder",
    "trizod": RESULTS / "human_proteome/UdonPred/trizod",
}

protein_mean_frames = []
for predictor, path in protein_mean_paths.items():
    predictions = load_predictions(path)
    predictor_rows = []
    for protein_id, record in predictions.items():
        finite_scores = record.scores[np.isfinite(record.scores)]
        if len(finite_scores):
            predictor_rows.append(
                {
                    "predictor": predictor,
                    "protein_id": protein_id,
                    "protein_mean": float(np.mean(finite_scores)),
                    "protein_length": len(finite_scores),
                }
            )
    protein_mean_frames.append(pd.DataFrame(predictor_rows))
    del predictions

protein_means = pd.concat(protein_mean_frames, ignore_index=True)

protein_mean_wide = protein_means.pivot(
    index="protein_id",
    columns="predictor",
    values="protein_mean",
)
comparison_predictors = [
    predictor
    for predictor in ["DisoFLAG", "DisorderUnetLM", "PUNCH2_light", "trizod"]
    if predictor in protein_mean_wide
]

top_overlap_rows = []
for predictor in comparison_predictors:
    pair = protein_mean_wide[["DisPredict3", predictor]].dropna()
    top_n = int(np.ceil(0.10 * len(pair)))
    dispredict3_top = set(pair["DisPredict3"].nlargest(top_n).index)
    predictor_top = set(pair[predictor].nlargest(top_n).index)
    shared_top = len(dispredict3_top & predictor_top)
    top_overlap_rows.append(
        {
            "predictor": predictor,
            "common_proteins": len(pair),
            "proteins_in_top_10_percent": top_n,
            "shared_top_10_percent_proteins": shared_top,
            "top_10_percent_overlap": 100 * shared_top / top_n,
        }
    )

dispredict3_top_overlap = pd.DataFrame(top_overlap_rows).sort_values(
    "top_10_percent_overlap",
    ascending=False,
)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(
    data=dispredict3_top_overlap,
    x="top_10_percent_overlap",
    y="predictor",
    color="tab:blue",
    ax=ax,
)
ax.axvline(10, color="black", linestyle="--", linewidth=1, label="Random expectation (10%)")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3)
ax.set_xlim(0, 100)
ax.set_xlabel("DisPredict3 top-10% proteins also in predictor top 10%")
ax.set_ylabel("")
ax.set_title("Agreement on the Most Disorder-Prone Proteins")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "dispredict3_top10_protein_overlap.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

display(dispredict3_top_overlap)
dispredict3_top_overlap.to_csv(
    OUTPUT_DIR / "dispredict3_top10_protein_overlap.csv",
    index=False,
)

#### Interpretation

Each bar shows the percentage of DisPredict3's top-10% proteins that the comparison predictor also places in its own top 10%. A higher bar therefore means stronger agreement about which complete proteins are particularly disorder-prone.

The dashed line marks the approximately 10% overlap expected by chance. The plot does not measure accuracy; it only makes the protein-level agreement question concrete.

## Mean Agreement per Predictor

For each predictor, average its pairwise Spearman correlations with every other predictor. High values indicate a predictor close to the shared signal of the complete predictor panel. Low values identify more independent predictors or outliers.

This is a consensus-proximity diagnostic, not an accuracy score: a predictor can disagree with the majority and still be correct.

In [ ]:
from IPython.display import display

agreement_columns = ["residue_spearman", "protein_spearman"]

agreement_by_predictor = pd.concat(
    [
        pairwise[["predictor_a", *agreement_columns]].rename(
            columns={"predictor_a": "predictor"}
        ),
        pairwise[["predictor_b", *agreement_columns]].rename(
            columns={"predictor_b": "predictor"}
        ),
    ],
    ignore_index=True,
)

mean_agreement = (
    agreement_by_predictor.groupby("predictor", as_index=False)
    .agg(
        mean_residue_spearman=("residue_spearman", "mean"),
        mean_protein_spearman=("protein_spearman", "mean"),
        compared_pairs=("residue_spearman", "size"),
    )
    .sort_values("mean_protein_spearman", ascending=False)
)

mean_agreement_long = mean_agreement.melt(
    id_vars=["predictor", "compared_pairs"],
    value_vars=["mean_residue_spearman", "mean_protein_spearman"],
    var_name="level",
    value_name="mean_spearman",
)
mean_agreement_long["level"] = mean_agreement_long["level"].map(
    {
        "mean_residue_spearman": "Residue level",
        "mean_protein_spearman": "Protein level",
    }
)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(
    data=mean_agreement_long,
    x="mean_spearman",
    y="predictor",
    hue="level",
    order=mean_agreement["predictor"],
    ax=ax,
)
ax.set_xlim(0, 1)
ax.set_xlabel("Mean pairwise Spearman correlation")
ax.set_ylabel("")
ax.set_title("Mean Agreement of Each Predictor With All Others")
ax.legend(title="")
fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "mean_agreement_per_predictor.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

display(mean_agreement)
mean_agreement.to_csv(
    OUTPUT_DIR / "mean_agreement_per_predictor.csv",
    index=False,
)

## Top-10% Protein Overlap Across All Predictors

For each predictor pair, select the 10% of common proteins with the highest mean disorder score and calculate how many proteins occur in both selections.

- 100%: identical top-10% selections
- approximately 10%: overlap expected from unrelated selections
- intermediate values: partial agreement about the most disorder-prone proteins

CheZOD and pLDDT are direction-corrected before ranking because lower native scores indicate more disorder.

In [ ]:
from itertools import combinations
from IPython.display import display

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.compare_predictors import load_predictions

predictor_paths = {
    name: ROOT / path
    for name, path in (value.split("=", 1) for value in predictor_args)
}
negated_predictors = {"chezod", "plddt", "adopt"}

protein_mean_series = {}
for predictor, path in predictor_paths.items():
    predictions = load_predictions(path)
    means = {}
    direction = -1.0 if predictor.lower() in negated_predictors else 1.0
    for protein_id, record in predictions.items():
        finite_scores = record.scores[np.isfinite(record.scores)]
        if len(finite_scores):
            means[protein_id] = direction * float(np.mean(finite_scores))
    protein_mean_series[predictor] = pd.Series(means, dtype=float)
    del predictions

all_protein_means = pd.DataFrame(protein_mean_series)
overlap_order = [predictor for predictor in predictor_order if predictor in all_protein_means]
top10_overlap_matrix = pd.DataFrame(
    np.nan,
    index=overlap_order,
    columns=overlap_order,
)
top10_overlap_details = []

for predictor in overlap_order:
    top10_overlap_matrix.loc[predictor, predictor] = 100.0

for predictor_a, predictor_b in combinations(overlap_order, 2):
    pair = all_protein_means[[predictor_a, predictor_b]].dropna()
    top_n = int(np.ceil(0.10 * len(pair)))
    top_a = set(pair[predictor_a].nlargest(top_n).index)
    top_b = set(pair[predictor_b].nlargest(top_n).index)
    shared = len(top_a & top_b)
    overlap_percent = 100 * shared / top_n
    top10_overlap_matrix.loc[predictor_a, predictor_b] = overlap_percent
    top10_overlap_matrix.loc[predictor_b, predictor_a] = overlap_percent
    top10_overlap_details.append(
        {
            "predictor_a": predictor_a,
            "predictor_b": predictor_b,
            "common_proteins": len(pair),
            "proteins_per_top_10_percent": top_n,
            "shared_top_10_percent_proteins": shared,
            "top_10_percent_overlap": overlap_percent,
        }
    )

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    top10_overlap_matrix,
    annot=True,
    fmt=".0f",
    cmap="viridis",
    vmin=0,
    vmax=100,
    square=True,
    cbar_kws={"label": "Shared top-10% proteins (%)"},
    ax=ax,
)
ax.set_title("Overlap of the Most Disorder-Prone Proteins")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "predictor_top10_protein_overlap_heatmap.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

udonpred_predictors = {"trizod", "chezod", "softdis", "atlas", "plddt", "disprot"}
external_predictors = [
    predictor for predictor in overlap_order if predictor not in udonpred_predictors
]
external_top10_overlap_matrix = top10_overlap_matrix.loc[
    external_predictors, external_predictors
]

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(
    external_top10_overlap_matrix,
    annot=True,
    fmt=".0f",
    cmap="viridis",
    vmin=0,
    vmax=100,
    square=True,
    cbar_kws={"label": "Shared top-10% proteins (%)"},
    ax=ax,
)
ax.set_title("Overlap of the Most Disorder-Prone Proteins\nExternal Predictors Only")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "external_predictor_top10_protein_overlap_heatmap.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()
external_top10_overlap_matrix.to_csv(
    OUTPUT_DIR / "external_predictor_top10_protein_overlap_matrix.csv"
)

top10_overlap_details = pd.DataFrame(top10_overlap_details).sort_values(
    "top_10_percent_overlap",
    ascending=False,
)
top10_overlap_matrix.to_csv(OUTPUT_DIR / "predictor_top10_protein_overlap_matrix.csv")
top10_overlap_details.to_csv(
    OUTPUT_DIR / "predictor_top10_protein_overlap_pairs.csv",
    index=False,
)
display(top10_overlap_details)

### How to Interpret Both Plots

Predictors with high mean agreement and consistently high top-10% overlap form the central consensus group. Predictors with low values in both plots are methodologically more independent. High mean Spearman but lower top-10% overlap means that the overall ranking is similar while the most extreme proteins differ. Lower residue-level than protein-level agreement indicates similar global protein assessments but different local disorder boundaries.

In the current panel, DisProt, TriZOD, PUNCH2_light, ADOPT, CheZOD, metapredict, SETH, ATLAS, pLDDT, and SoftDis form the main high-agreement group. ADOPT, metapredict, and SETH therefore follow the main shared disorder signal despite being external predictors. TriZOD and CheZOD have the strongest top-10% overlap (approximately 92%). IUPred3 is more independent, with lower mean agreement than the main group. DisoFLAG, DisorderUnetLM, and DisPredict3 form another distinct group, with top-10% overlaps of approximately 60-68% among themselves. DisPredict3 has the lowest mean agreement, especially at protein level.

Neither plot determines which predictor is correct; an independent benchmark is required for accuracy.

## Formal Predictor Clustering

The heatmaps above use a fixed display order. Here, average-linkage hierarchical clustering orders predictors from the distance `1 - Spearman`. The colored side bar marks UdonPred models, CAID-style external predictors, and classical external predictors. Clustering is performed separately for pooled residue-level Spearman and protein mean-score Spearman. It describes similarity, not benchmark accuracy.


In [ ]:
from IPython.display import Image, display

GLOBAL_DIR = OUTPUT_DIR / "global_behavior"
required_global_outputs = [
    GLOBAL_DIR / "predictor_clusters_residue.png",
    GLOBAL_DIR / "predictor_clusters_protein.png",
    GLOBAL_DIR / "protein_length_vs_mean_scores.png",
    GLOBAL_DIR / "predictor_qc.csv",
]
if not all(path.exists() for path in required_global_outputs):
    subprocess.run(
        [sys.executable, str(ROOT / "scripts/analyze_global_predictor_behavior.py")],
        cwd=ROOT,
        check=True,
    )

display(Image(filename=str(GLOBAL_DIR / "predictor_clusters_residue.png")))
display(Image(filename=str(GLOBAL_DIR / "predictor_clusters_protein.png")))
group_agreement = pd.read_csv(GLOBAL_DIR / "predictor_group_agreement.csv")
display(group_agreement)


### Cluster Interpretation

The expected method labels are informative but do not define perfectly separated clusters. The six UdonPred models have the strongest within-group agreement (mean residue Spearman 0.85; protein-level Spearman 0.92). DisPredict3, DisoFLAG, and DisorderUnetLM are adjacent at both analysis levels, while PUNCH2_light is closer to parts of the main consensus group. This supports a shared disorder signal plus method-specific subgroups; it does not show that one group is more accurate.


## Training Annotation and Predictor Behavior

For each UdonPred model pair, human-proteome predictor Spearman is compared with the existing exact-overlap annotation estimate. Continuous annotation pairs use Spearman; pairs involving binary DisProt use AUROC. These metrics are shown together only as pair-specific agreement indicators. Point size represents the number of compared annotation residues.


In [ ]:
display(Image(filename=str(GLOBAL_DIR / "annotation_vs_predictor_agreement.png")))
annotation_comparison = pd.read_csv(
    GLOBAL_DIR / "annotation_vs_predictor_agreement.csv"
)
annotation_summary = pd.read_csv(
    GLOBAL_DIR / "annotation_vs_predictor_summary.csv"
)
display(annotation_comparison)
display(annotation_summary)


### Annotation Comparison Interpretation

Among the nine available continuous annotation pairs, annotation agreement and human-proteome model agreement have a descriptive Spearman correlation of 0.60. This is compatible with annotation type influencing learned behavior, but it is not a significance test or causal result. Several points use only one or two overlapping proteins, the pairs are statistically dependent, and DisProt requires a different metric. The result should therefore be presented as a pattern that motivates interpretation, not as evidence of a universal annotation-performance relationship.


## Protein-Length Effects

Native score scales are unsuitable for cross-predictor variance. Each predictor's protein means are therefore converted to within-predictor percentiles. Predictor disagreement is the standard deviation of these percentiles for each protein. Pairwise agreement within a length class is Spearman over protein mean scores, so every protein has equal weight. This differs from pooled residue-level correlation, where long proteins contribute more observations.


In [ ]:
display(Image(filename=str(GLOBAL_DIR / "protein_length_effects.png")))
display(Image(filename=str(GLOBAL_DIR / "protein_length_vs_mean_scores.png")))
length_bins = pd.read_csv(GLOBAL_DIR / "length_bin_summary.csv")
length_effects = pd.read_csv(GLOBAL_DIR / "length_effects_by_predictor.csv")
display(length_bins)
display(length_effects.sort_values("length_vs_mean_score_spearman"))


### Length Interpretation

All four length classes are well populated (2,422 to 8,715 proteins). Predictor disagreement is highest for proteins shorter than 200 residues, followed by proteins of at least 1,000 residues; the middle classes are more consistent. Length effects are predictor-specific: DisPredict3, DisorderUnetLM, and DisoFLAG assign lower mean-score ranks to longer proteins, whereas IUPred3 shows the strongest positive relationship. Consequently, pooled residue-level and equal-protein analyses should both be reported.


## Methodological Quality Control

All scores are oriented so that higher means more disorder. CheZOD, pLDDT, and ADOPT are multiplied by -1 for rank-based comparisons. A universal threshold of 0.5 is not used: bounded scores are not automatically calibrated to the same biological decision boundary, so cross-predictor high-score analyses use within-predictor quantiles.


In [ ]:
predictor_qc = pd.read_csv(GLOBAL_DIR / "predictor_qc.csv")
qc_columns = [
    "predictor", "proteins", "common_proteins_with_reference",
    "length_mismatched_common_proteins",
    "overlap_sequence_mismatched_proteins",
    "score_transform", "cross_predictor_threshold_policy",
]
display(predictor_qc[qc_columns])


### QC Interpretation

UdonPred, IUPred3, SETH, DisorderUnetLM, DisoFLAG, PUNCH2_light, and DisPredict3 have no sequence or length mismatch among shared proteins. ADOPT is truncated for 2,317 proteins, as expected from its input limit, and should not be treated as full-length for those proteins. metapredict has 67 sequence mismatches that require caution for residue-position analyses. Pairwise CSV files report `residue_spearman` as one correlation over all matched residues and `protein_spearman` as a correlation over per-protein mean scores; neither is the mean of per-protein residue correlations.


## Local rq_5 Merged Sections

The following sections were preserved from the local `rq_5` branch during the `caid4` integration merge.


In [ ]:
agreement_long = pairwise.melt(
    id_vars=["predictor_a", "predictor_b", "common_proteins", "matched_residues"],
    value_vars=["residue_spearman", "protein_spearman"],
    var_name="level",
    value_name="spearman",
)
agreement_long["pair"] = [pair_label(a, b) for a, b in zip(agreement_long["predictor_a"], agreement_long["predictor_b"])]
agreement_long["level"] = agreement_long["level"].map(LEVEL_NAME)

# A full 55-pair bar plot is hard to read in a talk. Show the strongest and weakest residue-level pairs instead.
ranked_residue_pairs = (
    pairwise.assign(pair=lambda frame: [pair_label(a, b) for a, b in zip(frame["predictor_a"], frame["predictor_b"])])
    .sort_values("residue_spearman", ascending=False)
)
top_pairs = ranked_residue_pairs.head(12).copy()
bottom_pairs = ranked_residue_pairs.tail(12).sort_values("residue_spearman", ascending=True).copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharex=True)
for ax, data, title, color in [
    (axes[0], top_pairs, "Strongest agreement", "#2a9d8f"),
    (axes[1], bottom_pairs, "Weakest agreement", "#e76f51"),
]:
    sns.barplot(data=data, x="residue_spearman", y="pair", ax=ax, color=color)
    ax.set_title(title)
    ax.set_xlabel("Residue-level Spearman correlation")
    ax.set_ylabel("")
    ax.set_xlim(0, 1)
    add_bar_labels(ax, fontsize=9)

fig.suptitle("Most and least similar predictor pairs on the human proteome", y=1.03, fontsize=20)
add_note(fig, "Spearman correlation measures whether two predictors rank residues similarly. It is not an accuracy score.")
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(OUTPUT_DIR / "predictor_pairwise_spearman_bars.png", dpi=200, bbox_inches="tight")
plt.show()


### Predictor Clusters

The heatmaps above show individual pairwise values. Clustering the residue-level agreement matrix gives a compact view of which predictors share the same human-proteome signal. This is the main result to interpret before looking at annotation-source consistency.

In [ ]:
cluster_matrix = display_matrix(residue_spearman_matrix.astype(float).copy())
cluster_grid = sns.clustermap(
    cluster_matrix,
    cmap="viridis",
    vmin=np.nanmin(cluster_matrix.values),
    vmax=1,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    figsize=(10, 10),
    cbar_kws={"label": "Residue Spearman"},
)
cluster_grid.fig.suptitle("Predictors cluster by shared human-proteome signal", y=1.03, fontsize=18)
cluster_grid.ax_heatmap.set_xlabel("")
cluster_grid.ax_heatmap.set_ylabel("")
cluster_grid.ax_heatmap.tick_params(axis="x", rotation=45)
cluster_grid.ax_heatmap.tick_params(axis="y", rotation=0)
add_note(cluster_grid.fig, "Predictors placed close together have more similar residue-level score patterns.")
cluster_grid.savefig(OUTPUT_DIR / "predictor_residue_spearman_clustermap.png", dpi=200, bbox_inches="tight")
plt.show()

cluster_order = [cluster_matrix.index[i] for i in cluster_grid.dendrogram_row.reordered_ind]
cluster_summary = pd.DataFrame({"cluster_order_residue_spearman": cluster_order})
cluster_summary


## Predictor Agreement vs Annotation-Source Consistency

The direct comparison to experimental annotation agreement is restricted to the seven UdonPred dataset-specific models. External predictors are kept in the predictor-agreement analysis, but they do not have a one-to-one annotation source in this project setup.

The annotation agreement values are reused from the experimental annotation-ceiling analysis. Positive gaps mean that two trained predictors agree more strongly on the independent human-proteome panel than the corresponding experimental annotation sources agree on their own overlapping annotated proteins. This is an interpretability comparison, not a human-proteome accuracy benchmark.

In [ ]:
ANNOTATION_CEILING_CSV = RESULTS / "annotation_ceiling" / "annotation_ceiling_summary.csv"
ANNOTATION_CEILING_MMSEQS_CSV = RESULTS / "annotation_ceiling_mmseqs" / "annotation_ceiling_summary.csv"
MMSEQS_MIN_IDENTITY = 90.0
UDONPRED_DATASETS = ["trizod", "chezod", "softdis", "pdbflex", "atlas", "plddt", "disprot"]

# Annotation-ceiling files summarize how the original experimental label sources
# agree with each other on their overlapping proteins/residues.
# These are separate from the human-proteome prediction outputs below.
annotation_ceiling = pd.read_csv(ANNOTATION_CEILING_CSV)
annotation_ceiling_mmseqs_raw = pd.read_csv(ANNOTATION_CEILING_MMSEQS_CSV)

# MMseqs is kept as a sensitivity analysis: it increases annotation overlap by
# allowing highly similar sequence matches instead of only exact overlaps.
annotation_ceiling_mmseqs = annotation_ceiling_mmseqs_raw[
    (annotation_ceiling_mmseqs_raw["match_mode"] == "mmseqs")
    & (annotation_ceiling_mmseqs_raw["min_identity"] == MMSEQS_MIN_IDENTITY)
].copy()

# Use an order-independent key so "trizod vs chezod" and "chezod vs trizod"
# map to the same predictor/annotation pair.
def normalized_pair_key(left, right):
    return "__".join(sorted([str(left).lower(), str(right).lower()]))

def build_annotation_pair_summary(annotation_frame, column_prefix="annotation"):
    # Collapse the long annotation-ceiling table to one row per dataset pair.
    # Keep both agreement values and overlap counts because small overlaps make
    # Spearman estimates less stable.
    annotation_frame = annotation_frame.copy()
    annotation_frame["pair_key"] = [
        normalized_pair_key(left, right)
        for left, right in zip(annotation_frame["dataset_a"], annotation_frame["dataset_b"])
    ]
    counts = (
        annotation_frame.sort_values(["pair_key", "metric"])
        .groupby("pair_key", as_index=False)
        .agg(
            annotation_dataset_a=("dataset_a", "first"),
            annotation_dataset_b=("dataset_b", "first"),
            annotation_match_mode=("match_mode", "first"),
            n_annotation_proteins=("n_proteins_overlap", "max"),
            n_annotation_residues=("n_residues_compared", "max"),
        )
    )
    value_frame = annotation_frame[annotation_frame["metric"].isin(["spearman", "protein_spearman"])]
    values = (
        value_frame.pivot_table(index="pair_key", columns="metric", values="value", aggfunc="first")
        .reset_index()
    )
    if "spearman" not in values.columns:
        values["spearman"] = np.nan
    if "protein_spearman" not in values.columns:
        values["protein_spearman"] = np.nan
    values = values.rename(
        columns={
            "spearman": f"{column_prefix}_residue_spearman",
            "protein_spearman": f"{column_prefix}_protein_spearman",
        }
    )
    return counts.merge(values, on="pair_key", how="left")

# Only UdonPred dataset-specific models have a direct underlying annotation
# source in this project, so external predictors are excluded from this join.
udon_pairwise = pairwise[
    pairwise["predictor_a"].str.lower().isin(UDONPRED_DATASETS)
    & pairwise["predictor_b"].str.lower().isin(UDONPRED_DATASETS)
].copy()
udon_pairwise["pair_key"] = [
    normalized_pair_key(left, right)
    for left, right in zip(udon_pairwise["predictor_a"], udon_pairwise["predictor_b"])
]
udon_pairwise["pair"] = udon_pairwise["predictor_a"] + " vs " + udon_pairwise["predictor_b"]

annotation_pair_summary = build_annotation_pair_summary(annotation_ceiling, "annotation")
annotation_pair_summary_mmseqs = build_annotation_pair_summary(annotation_ceiling_mmseqs, "mmseqs_annotation")

# Join each predictor pair to the corresponding experimental annotation pair.
# Predictor agreement: model outputs on the human proteome.
# Annotation agreement: original experimental labels on dataset overlap.
udon_predictor_vs_annotation = udon_pairwise.merge(
    annotation_pair_summary,
    on="pair_key",
    how="left",
)
udon_predictor_vs_annotation_sensitivity = udon_pairwise.merge(
    annotation_pair_summary_mmseqs,
    on="pair_key",
    how="left",
)

# Agreement gap quantifies whether model-output agreement exceeds the
# experimental annotation baseline. Positive gap = stronger predictor
# convergence than annotation agreement. It is not an accuracy claim.
udon_predictor_vs_annotation["residue_gap"] = (
    udon_predictor_vs_annotation["residue_spearman"]
    - udon_predictor_vs_annotation["annotation_residue_spearman"]
)
udon_predictor_vs_annotation["protein_gap"] = (
    udon_predictor_vs_annotation["protein_spearman"]
    - udon_predictor_vs_annotation["annotation_protein_spearman"]
)
udon_predictor_vs_annotation_sensitivity["residue_gap"] = (
    udon_predictor_vs_annotation_sensitivity["residue_spearman"]
    - udon_predictor_vs_annotation_sensitivity["mmseqs_annotation_residue_spearman"]
)
udon_predictor_vs_annotation_sensitivity["protein_gap"] = (
    udon_predictor_vs_annotation_sensitivity["protein_spearman"]
    - udon_predictor_vs_annotation_sensitivity["mmseqs_annotation_protein_spearman"]
)

# Flag annotation pairs with little or no experimental overlap so their gaps
# can be interpreted cautiously in tables and plots.
udon_predictor_vs_annotation["annotation_overlap_confidence"] = np.select(
    [
        udon_predictor_vs_annotation["n_annotation_residues"].fillna(0) == 0,
        (udon_predictor_vs_annotation["n_annotation_residues"].fillna(0) < 100)
        | (udon_predictor_vs_annotation["n_annotation_proteins"].fillna(0) < 3),
    ],
    ["no overlap", "low overlap"],
    default="usable overlap",
)
udon_predictor_vs_annotation_sensitivity["annotation_overlap_confidence"] = np.select(
    [
        udon_predictor_vs_annotation_sensitivity["n_annotation_residues"].fillna(0) == 0,
        (udon_predictor_vs_annotation_sensitivity["n_annotation_residues"].fillna(0) < 100)
        | (udon_predictor_vs_annotation_sensitivity["n_annotation_proteins"].fillna(0) < 3),
    ],
    ["no overlap", "low overlap"],
    default="usable overlap",
)

comparison_columns = [
    "pair",
    "common_proteins",
    "matched_residues",
    "residue_spearman",
    "annotation_residue_spearman",
    "residue_gap",
    "protein_spearman",
    "annotation_protein_spearman",
    "protein_gap",
    "n_annotation_proteins",
    "n_annotation_residues",
    "annotation_overlap_confidence",
]
udon_predictor_vs_annotation[comparison_columns].sort_values("residue_gap", ascending=False)


### Human-Proteome Coverage Caveat

The human proteome is independent of the seven UdonPred annotation datasets. The table below does not provide annotation agreement. It shows how much of each UdonPred test set has an exact sequence match in the human proteome, which helps judge whether a model's annotation source is directly human-proteome-like or mostly out-of-panel.

In [ ]:
HUMAN_COVERAGE_CSV = RESULTS / "human_proteome_annotation_ceiling" / "human_proteome_overlap_summary.csv"

if HUMAN_COVERAGE_CSV.exists():
    human_coverage = pd.read_csv(HUMAN_COVERAGE_CSV)
else:
    human_coverage = pd.DataFrame(
        columns=[
            "dataset",
            "n_udonpred_test_proteins",
            "n_matched_human_proteins",
            "protein_overlap_fraction",
            "n_valid_annotated_residues",
            "n_matched_valid_annotated_residues",
            "valid_annotated_residue_overlap_fraction",
        ]
    )

coverage_display = human_coverage[
    [
        "dataset",
        "n_udonpred_test_proteins",
        "n_matched_human_proteins",
        "protein_overlap_fraction",
        "n_valid_annotated_residues",
        "n_matched_valid_annotated_residues",
        "valid_annotated_residue_overlap_fraction",
    ]
].sort_values("valid_annotated_residue_overlap_fraction", ascending=False)
coverage_display

In [ ]:
if not human_coverage.empty:
    plot_coverage = human_coverage.sort_values("valid_annotated_residue_overlap_fraction", ascending=False).copy()
    plot_coverage["dataset_label"] = plot_coverage["dataset"].map(display_name)
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(
        data=plot_coverage,
        x="valid_annotated_residue_overlap_fraction",
        y="dataset_label",
        color="#3d8f73",
        ax=ax,
    )
    ax.set_xlim(0, 1)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_xlabel("Exact human-proteome overlap among valid annotated residues")
    ax.set_ylabel("UdonPred annotation source")
    ax.set_title("Which annotation sources are directly represented in the human proteome?")
    add_bar_labels(ax, fmt="{:.0%}", x_offset=0.01)
    add_note(fig, "This is a coverage/provenance check, not an annotation-agreement score.")
    plt.tight_layout(rect=(0, 0.06, 1, 1))
    plt.savefig(OUTPUT_DIR / "udon_annotation_source_human_coverage.png", dpi=200, bbox_inches="tight")
    plt.show()

if not human_coverage.empty:
    coverage_lookup = human_coverage.set_index("dataset")
    udon_predictor_vs_annotation["min_pair_human_valid_residue_overlap"] = [
        min(
            coverage_lookup.loc[str(left).lower(), "valid_annotated_residue_overlap_fraction"],
            coverage_lookup.loc[str(right).lower(), "valid_annotated_residue_overlap_fraction"],
        )
        for left, right in zip(
            udon_predictor_vs_annotation["predictor_a"],
            udon_predictor_vs_annotation["predictor_b"],
        )
    ]
    udon_predictor_vs_annotation["human_coverage_confidence"] = pd.cut(
        udon_predictor_vs_annotation["min_pair_human_valid_residue_overlap"],
        bins=[-0.001, 0.0, 0.05, 0.2, 1.0],
        labels=["no exact human overlap", "low exact human overlap", "moderate exact human overlap", "higher exact human overlap"],
    )
else:
    udon_predictor_vs_annotation["min_pair_human_valid_residue_overlap"] = np.nan
    udon_predictor_vs_annotation["human_coverage_confidence"] = "coverage file missing"

coverage_context_columns = comparison_columns + [
    "min_pair_human_valid_residue_overlap",
    "human_coverage_confidence",
]
udon_predictor_vs_annotation[coverage_context_columns].sort_values("residue_gap", ascending=False)


In [ ]:
mmseqs_sensitivity_comparison = udon_predictor_vs_annotation[
    [
        "pair_key",
        "pair",
        "residue_spearman",
        "annotation_residue_spearman",
        "residue_gap",
        "protein_spearman",
        "annotation_protein_spearman",
        "protein_gap",
        "n_annotation_proteins",
        "n_annotation_residues",
    ]
].merge(
    udon_predictor_vs_annotation_sensitivity[
        [
            "pair_key",
            "mmseqs_annotation_residue_spearman",
            "mmseqs_annotation_protein_spearman",
            "residue_gap",
            "protein_gap",
            "n_annotation_proteins",
            "n_annotation_residues",
        ]
    ].rename(
        columns={
            "residue_gap": "mmseqs_residue_gap",
            "protein_gap": "mmseqs_protein_gap",
            "n_annotation_proteins": "mmseqs_n_annotation_proteins",
            "n_annotation_residues": "mmseqs_n_annotation_residues",
        }
    ),
    on="pair_key",
    how="left",
)
mmseqs_sensitivity_comparison["mmseqs_min_identity"] = MMSEQS_MIN_IDENTITY
mmseqs_sensitivity_comparison["annotation_residue_spearman_change"] = (
    mmseqs_sensitivity_comparison["mmseqs_annotation_residue_spearman"]
    - mmseqs_sensitivity_comparison["annotation_residue_spearman"]
)
mmseqs_sensitivity_comparison["annotation_protein_spearman_change"] = (
    mmseqs_sensitivity_comparison["mmseqs_annotation_protein_spearman"]
    - mmseqs_sensitivity_comparison["annotation_protein_spearman"]
)

mmseqs_display_columns = [
    "pair",
    "residue_spearman",
    "annotation_residue_spearman",
    "mmseqs_annotation_residue_spearman",
    "residue_gap",
    "mmseqs_residue_gap",
    "protein_spearman",
    "annotation_protein_spearman",
    "mmseqs_annotation_protein_spearman",
    "protein_gap",
    "mmseqs_protein_gap",
    "n_annotation_proteins",
    "n_annotation_residues",
    "mmseqs_n_annotation_proteins",
    "mmseqs_n_annotation_residues",
    "annotation_residue_spearman_change",
    "annotation_protein_spearman_change",
]
mmseqs_sensitivity_comparison[mmseqs_display_columns].sort_values(
    "annotation_residue_spearman_change",
    ascending=False,
)


In [ ]:
plot_data = udon_predictor_vs_annotation.dropna(
    subset=["annotation_residue_spearman", "residue_spearman"]
).copy()
plot_data = plot_data.sort_values("residue_gap", ascending=False).reset_index(drop=True)
plot_data["pair_label"] = [pair_label(pair) for pair in plot_data["pair"]]
plot_data["plot_id"] = np.arange(1, len(plot_data) + 1)
plot_data["overlap_size"] = np.clip(np.log10(plot_data["n_annotation_residues"].fillna(1) + 1) * 55, 80, 320)

fig, ax = plt.subplots(figsize=(9, 8))
limit_min = min(plot_data["annotation_residue_spearman"].min(), plot_data["residue_spearman"].min(), -0.2)
limit_max = 1.0
max_abs_gap = max(abs(plot_data["residue_gap"].min()), abs(plot_data["residue_gap"].max()))

scatter = ax.scatter(
    plot_data["annotation_residue_spearman"],
    plot_data["residue_spearman"],
    c=plot_data["residue_gap"],
    cmap="coolwarm",
    vmin=-max_abs_gap,
    vmax=max_abs_gap,
    s=plot_data["overlap_size"],
    edgecolor="white",
    linewidth=1.2,
    alpha=0.9,
)

ax.plot([limit_min, limit_max], [limit_min, limit_max], color="0.25", linestyle="--", linewidth=1.5)
ax.fill_between(
    [limit_min, limit_max],
    [limit_min, limit_max],
    [limit_max, limit_max],
    color="#277da1",
    alpha=0.08,
)
ax.text(
    0.03,
    0.93,
    "above line:\npredictors agree more",
    transform=ax.transAxes,
    color="#277da1",
    fontsize=11,
    ha="left",
    va="top",
)

for row in plot_data.itertuples(index=False):
    ax.annotate(
        str(row.plot_id),
        (row.annotation_residue_spearman, row.residue_spearman),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=10,
        weight="bold",
    )

cbar = fig.colorbar(scatter, ax=ax, shrink=0.82)
cbar.set_label("Gap: predictor agreement - annotation agreement")
ax.set_xlim(limit_min, limit_max + 0.03)
ax.set_ylim(limit_min, limit_max + 0.03)
ax.set_xlabel("Annotation-source agreement on overlapping labeled proteins")
ax.set_ylabel("Predictor agreement on the human proteome")
ax.set_title("Do trained predictors agree more than their source annotations?")
ax.grid(True, color="0.9")
add_note(fig, "Each point is one UdonPred head pair. Larger points have more labeled residues in the annotation comparison.")
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(OUTPUT_DIR / "udon_predictor_vs_annotation_residue_scatter.png", dpi=200, bbox_inches="tight")
plt.show()

plot_data[
    [
        "plot_id",
        "pair_label",
        "annotation_residue_spearman",
        "residue_spearman",
        "residue_gap",
        "n_annotation_proteins",
        "n_annotation_residues",
        "annotation_overlap_confidence",
    ]
].rename(
    columns={
        "plot_id": "#",
        "pair_label": "pair",
        "annotation_residue_spearman": "annotation agreement",
        "residue_spearman": "predictor agreement",
        "residue_gap": "gap",
        "n_annotation_proteins": "annotation proteins",
        "n_annotation_residues": "annotation residues",
        "annotation_overlap_confidence": "overlap confidence",
    }
)


In [ ]:
plot_data = mmseqs_sensitivity_comparison.dropna(
    subset=["mmseqs_annotation_residue_spearman", "residue_spearman"]
).copy()
plot_data = plot_data.sort_values("mmseqs_residue_gap", ascending=False).reset_index(drop=True)
plot_data["pair_label"] = [pair_label(pair) for pair in plot_data["pair"]]
plot_data["plot_id"] = np.arange(1, len(plot_data) + 1)
plot_data["overlap_size"] = np.clip(np.log10(plot_data["mmseqs_n_annotation_residues"].fillna(1) + 1) * 55, 80, 320)

fig, ax = plt.subplots(figsize=(9, 8))
limit_min = min(plot_data["mmseqs_annotation_residue_spearman"].min(), plot_data["residue_spearman"].min(), -0.2)
limit_max = 1.0
max_abs_gap = max(abs(plot_data["mmseqs_residue_gap"].min()), abs(plot_data["mmseqs_residue_gap"].max()))

scatter = ax.scatter(
    plot_data["mmseqs_annotation_residue_spearman"],
    plot_data["residue_spearman"],
    c=plot_data["mmseqs_residue_gap"],
    cmap="coolwarm",
    vmin=-max_abs_gap,
    vmax=max_abs_gap,
    s=plot_data["overlap_size"],
    edgecolor="white",
    linewidth=1.2,
    alpha=0.9,
)

ax.plot([limit_min, limit_max], [limit_min, limit_max], color="0.25", linestyle="--", linewidth=1.5)
ax.fill_between(
    [limit_min, limit_max],
    [limit_min, limit_max],
    [limit_max, limit_max],
    color="#277da1",
    alpha=0.08,
)
ax.text(
    0.03,
    0.93,
    "above line:\npredictors agree more",
    transform=ax.transAxes,
    color="#277da1",
    fontsize=11,
    ha="left",
    va="top",
)

for row in plot_data.itertuples(index=False):
    ax.annotate(
        str(row.plot_id),
        (row.mmseqs_annotation_residue_spearman, row.residue_spearman),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=10,
        weight="bold",
    )

cbar = fig.colorbar(scatter, ax=ax, shrink=0.82)
cbar.set_label("Gap: predictor agreement - MMseqs annotation agreement")
ax.set_xlim(limit_min, limit_max + 0.03)
ax.set_ylim(limit_min, limit_max + 0.03)
ax.set_xlabel(f"Annotation-source agreement with MMseqs matches ({MMSEQS_MIN_IDENTITY:.0f}% identity)")
ax.set_ylabel("Predictor agreement on the human proteome")
ax.set_title("Sensitivity check with expanded annotation overlap")
ax.grid(True, color="0.9")
add_note(fig, "Each point is one UdonPred head pair. Larger points have more MMseqs-matched annotation residues.")
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(OUTPUT_DIR / "udon_predictor_vs_mmseqs_annotation_residue_agreement.png", dpi=200, bbox_inches="tight")
plt.show()

plot_data[
    [
        "plot_id",
        "pair_label",
        "mmseqs_annotation_residue_spearman",
        "residue_spearman",
        "mmseqs_residue_gap",
        "mmseqs_n_annotation_proteins",
        "mmseqs_n_annotation_residues",
    ]
].rename(
    columns={
        "plot_id": "#",
        "pair_label": "pair",
        "mmseqs_annotation_residue_spearman": "MMseqs annotation agreement",
        "residue_spearman": "predictor agreement",
        "mmseqs_residue_gap": "gap",
        "mmseqs_n_annotation_proteins": "MMseqs annotation proteins",
        "mmseqs_n_annotation_residues": "MMseqs annotation residues",
    }
)


In [ ]:
gap_data = udon_predictor_vs_annotation.dropna(subset=["residue_gap"]).sort_values(
    "residue_gap",
    ascending=False,
).copy()
gap_data["pair_label"] = [pair_label(pair) for pair in gap_data["pair"]]

fig, ax = plt.subplots(figsize=(11, 8))
sns.barplot(
    data=gap_data,
    x="residue_gap",
    y="pair_label",
    hue="annotation_overlap_confidence",
    order=gap_data["pair_label"].tolist(),
    dodge=False,
    palette=CONFIDENCE_PALETTE,
    ax=ax,
)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Agreement gap: predictor pair minus annotation-source pair")
ax.set_ylabel("UdonPred head pair")
ax.set_title("Where do predictors agree more than their annotation sources?")
ax.legend(title="Labeled overlap used for annotation comparison", loc="lower right", frameon=True)
add_bar_labels(ax, fontsize=9)
add_note(fig, "Bars to the right of zero mean predictor outputs are more similar than the underlying annotation sources.")
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(OUTPUT_DIR / "udon_predictor_annotation_residue_gap.png", dpi=200, bbox_inches="tight")
plt.show()

protein_gap_data = udon_predictor_vs_annotation.dropna(subset=["protein_gap"]).sort_values(
    "protein_gap",
    ascending=False,
).copy()
protein_gap_data["pair_label"] = [pair_label(pair) for pair in protein_gap_data["pair"]]
fig, ax = plt.subplots(figsize=(11, 7))
sns.barplot(
    data=protein_gap_data,
    x="protein_gap",
    y="pair_label",
    hue="annotation_overlap_confidence",
    order=protein_gap_data["pair_label"].tolist(),
    dodge=False,
    palette=CONFIDENCE_PALETTE,
    ax=ax,
)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Protein-level agreement gap")
ax.set_ylabel("UdonPred head pair")
ax.set_title("Protein-level predictor agreement vs annotation-source consistency")
ax.legend(title="Labeled overlap", loc="lower right", frameon=True)
add_bar_labels(ax, fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "udon_predictor_annotation_protein_gap.png", dpi=200, bbox_inches="tight")
plt.show()

exact_gap_frame = udon_predictor_vs_annotation[
    ["pair", "residue_gap", "protein_gap"]
].assign(annotation_match="exact")
mmseqs_gap_frame = udon_predictor_vs_annotation_sensitivity[
    ["pair", "residue_gap", "protein_gap"]
].assign(annotation_match=f"MMseqs >= {MMSEQS_MIN_IDENTITY:.0f}%")
gap_comparison = pd.concat([exact_gap_frame, mmseqs_gap_frame], ignore_index=True)
gap_comparison["pair_label"] = [pair_label(pair) for pair in gap_comparison["pair"]]

def keep_pairs_with_exact_and_mmseqs(frame, value_column):
    available = frame.dropna(subset=[value_column]).copy()
    complete_pairs = (
        available.groupby("pair")["annotation_match"]
        .nunique()
        .loc[lambda counts: counts == 2]
        .index
    )
    return available[available["pair"].isin(complete_pairs)].copy()

residue_gap_comparison = keep_pairs_with_exact_and_mmseqs(gap_comparison, "residue_gap")
residue_pair_order = (
    residue_gap_comparison.groupby("pair_label")["residue_gap"]
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
fig, ax = plt.subplots(figsize=(11, 8))
sns.barplot(
    data=residue_gap_comparison,
    x="residue_gap",
    y="pair_label",
    hue="annotation_match",
    order=residue_pair_order,
    dodge=True,
    palette={"exact": "0.45", f"MMseqs >= {MMSEQS_MIN_IDENTITY:.0f}%": "#2a9d8f"},
    ax=ax,
)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Agreement gap: predictor pair minus annotation-source pair")
ax.set_ylabel("UdonPred head pair")
ax.set_title("Does the conclusion change when annotation overlap is expanded?")
ax.legend(title="Annotation-source match rule", frameon=True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "udon_predictor_annotation_residue_gap_exact_vs_mmseqs.png", dpi=200, bbox_inches="tight")
plt.show()

protein_gap_comparison = keep_pairs_with_exact_and_mmseqs(gap_comparison, "protein_gap")
protein_pair_order = (
    protein_gap_comparison.groupby("pair_label")["protein_gap"]
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
if protein_gap_comparison["annotation_match"].nunique() > 1:
    fig, ax = plt.subplots(figsize=(11, 7))
    sns.barplot(
        data=protein_gap_comparison,
        x="protein_gap",
        y="pair_label",
        hue="annotation_match",
        order=protein_pair_order,
        dodge=True,
        palette={"exact": "0.45", f"MMseqs >= {MMSEQS_MIN_IDENTITY:.0f}%": "#2a9d8f"},
        ax=ax,
    )
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("Protein-level agreement gap")
    ax.set_ylabel("UdonPred head pair")
    ax.set_title("Protein-level gap: exact vs similar-sequence annotation matches")
    ax.legend(title="Annotation-source match rule", frameon=True)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "udon_predictor_annotation_protein_gap_exact_vs_mmseqs.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print(
        "No MMseqs protein-level annotation gaps available yet. "
        "Regenerate results/annotation_ceiling_mmseqs with the updated estimate_annotation_ceiling.py "
        "to create the protein exact-vs-MMseqs gap plot."
    )


## `pdbflex` Disagreement

The pairwise agreement table suggests that `pdbflex` is the most distinct UdonPred head on the human proteome. This section quantifies that directly by comparing each predictor's average agreement with `pdbflex` against its average agreement with all non-`pdbflex` predictors.

In [ ]:
def mean_agreement_with(target, value_column="residue_spearman"):
    rows = pairwise[(pairwise["predictor_a"] == target) | (pairwise["predictor_b"] == target)].copy()
    rows["other_predictor"] = np.where(rows["predictor_a"] == target, rows["predictor_b"], rows["predictor_a"])
    return rows[["other_predictor", value_column, "protein_spearman", "residue_zscore_mae"]].sort_values(value_column)

pdbflex_agreement = mean_agreement_with("pdbflex")
non_pdbflex_pairwise = pairwise[
    (pairwise["predictor_a"] != "pdbflex") & (pairwise["predictor_b"] != "pdbflex")
]
pdbflex_summary = pd.DataFrame(
    {
        "comparison_group": ["pairs with pdbflex", "pairs without pdbflex"],
        "mean_residue_spearman": [
            pdbflex_agreement["residue_spearman"].mean(),
            non_pdbflex_pairwise["residue_spearman"].mean(),
        ],
        "median_residue_spearman": [
            pdbflex_agreement["residue_spearman"].median(),
            non_pdbflex_pairwise["residue_spearman"].median(),
        ],
        "mean_protein_spearman": [
            pdbflex_agreement["protein_spearman"].mean(),
            non_pdbflex_pairwise["protein_spearman"].mean(),
        ],
        "median_protein_spearman": [
            pdbflex_agreement["protein_spearman"].median(),
            non_pdbflex_pairwise["protein_spearman"].median(),
        ],
    }
)
pdbflex_summary

In [ ]:
pdbflex_plot_data = pdbflex_agreement.copy()
pdbflex_plot_data["other_predictor_label"] = pdbflex_plot_data["other_predictor"].map(display_name)
pdbflex_plot_data = pdbflex_plot_data.sort_values("residue_spearman")

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
sns.barplot(
    data=pdbflex_plot_data,
    x="residue_spearman",
    y="other_predictor_label",
    ax=axes[0],
    color="#f4a261",
)
axes[0].set_title("Residue-level similarity to PDBFlex")
axes[0].set_xlabel("Spearman correlation")
axes[0].set_ylabel("Compared predictor")
axes[0].set_xlim(0, 1)
add_bar_labels(axes[0], fontsize=9)

sns.barplot(
    data=pdbflex_plot_data,
    x="protein_spearman",
    y="other_predictor_label",
    ax=axes[1],
    color="#7b2cbf",
)
axes[1].set_title("Protein-level similarity to PDBFlex")
axes[1].set_xlabel("Spearman correlation")
axes[1].set_ylabel("")
axes[1].set_xlim(0, 1)
add_bar_labels(axes[1], fontsize=9)

fig.suptitle("PDBFlex behaves differently from most disorder-focused predictors", y=1.03, fontsize=20)
add_note(fig, "Lower bars indicate pairs where PDBFlex ranks the human proteome differently.")
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(OUTPUT_DIR / "pdbflex_agreement_profile.png", dpi=200, bbox_inches="tight")
plt.show()

pdbflex_agreement


## External Predictors

The external methods remain useful as reference predictors on the same human-proteome panel. They are not assigned an annotation-ceiling baseline here because their training data do not map cleanly to one of the seven UdonPred annotation sources.


In [ ]:
external_predictors = sorted(
    (set(pairwise["predictor_a"]) | set(pairwise["predictor_b"])) - set(UDONPRED_DATASETS)
)
external_pairwise = pairwise[
    pairwise["predictor_a"].isin(external_predictors)
    | pairwise["predictor_b"].isin(external_predictors)
].copy()
external_pairwise["pair"] = external_pairwise["predictor_a"] + " vs " + external_pairwise["predictor_b"]
external_pairwise.sort_values("residue_spearman", ascending=False)[
    ["pair", "common_proteins", "matched_residues", "residue_spearman", "protein_spearman", "residue_zscore_mae"]
]


### External Predictors vs UdonPred Consensus

To include external predictors without assigning them an artificial annotation baseline, compare each external method to a scale-normalized consensus of the seven UdonPred dataset-specific models. UdonPred models are also compared to a leave-one-out UdonPred consensus to provide context for how close a method is to the shared UdonPred signal.

Consensus = What is the common signal that the seven UdonPred models agree on, on average?

The consensus is built after all predictor outputs are put into the same score direction and z-score normalized. It is not a ground truth label. It is an internal reference signal: the residue-wise average of the normalized UdonPred model outputs.

For UdonPred models, the target model is excluded from its own consensus (leave-one-out). For external predictors, the target is compared to the full seven-model UdonPred consensus because there is no direct annotation-ceiling baseline for those external methods.

In [ ]:
CONSENSUS_CSV = OUTPUT_DIR / "predictor_vs_udonpred_consensus.csv"

if CONSENSUS_CSV.exists():
    predictor_vs_udon_consensus = pd.read_csv(CONSENSUS_CSV)
else:
    missing_message = (
        f"Missing {CONSENSUS_CSV}. Recreate raw predictor outputs under "
        "results/human_proteome/ and rerun the consensus calculation, or restore "
        "the saved predictor_vs_udonpred_consensus.csv file."
    )
    raise FileNotFoundError(missing_message)

predictor_vs_udon_consensus.sort_values(
    "residue_spearman_vs_udon_consensus",
    ascending=False,
)


In [ ]:
consensus_plot_data = predictor_vs_udon_consensus.melt(
    id_vars=["predictor", "comparison_group", "common_proteins", "matched_residues"],
    value_vars=["residue_spearman_vs_udon_consensus", "protein_spearman_vs_udon_consensus"],
    var_name="level",
    value_name="spearman_vs_consensus",
)
consensus_plot_data["level"] = consensus_plot_data["level"].map(LEVEL_NAME)
consensus_plot_data["predictor_label"] = np.where(
    consensus_plot_data["comparison_group"] == "external predictor",
    consensus_plot_data["predictor"].map(display_name) + " (external)",
    consensus_plot_data["predictor"].map(display_name) + " (UdonPred)",
)
order = (
    consensus_plot_data.groupby("predictor_label")["spearman_vs_consensus"]
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(
    data=consensus_plot_data,
    x="spearman_vs_consensus",
    y="predictor_label",
    hue="level",
    order=order,
    palette={"Residue-level": "#277da1", "Protein-level": "#90be6d"},
    ax=ax,
)
ax.set_xlabel("Spearman correlation with the average UdonPred signal")
ax.set_ylabel("Predictor")
ax.set_title("How close is each method to the shared UdonPred consensus?")
ax.set_xlim(0, 1)
ax.legend(title="Comparison level", loc="lower right", frameon=True)
add_note(fig, "External predictors are compared to the UdonPred consensus without assigning them an annotation-source ceiling.")
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(OUTPUT_DIR / "predictor_vs_udonpred_consensus.png", dpi=200, bbox_inches="tight")
plt.show()


## Contested Regions

The contested-region table ranks fixed windows by disagreement after z-score scaling predictor outputs. These regions are candidates for follow-up inspection rather than model-performance errors, because no human-proteome ground truth is assumed here.

In [ ]:
contested_display_columns = [
    "protein_id",
    "start_residue",
    "end_residue",
    "mean_std",
    "max_std",
    "max_range",
    "max_disagreement_residue",
    "mean_score",
]
contested_top = contested_regions.sort_values(["mean_std", "max_std"], ascending=False).head(15).copy()
contested_top["region"] = (
    contested_top["protein_id"].astype(str)
    + ":"
    + contested_top["start_residue"].astype(str)
    + "-"
    + contested_top["end_residue"].astype(str)
)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(
    data=contested_top,
    x="mean_std",
    y="region",
    order=contested_top["region"].tolist(),
    color="#d62828",
    ax=ax,
)
ax.set_xlabel("Average disagreement between predictors in this 30-residue window")
ax.set_ylabel("Human protein region")
ax.set_title("Regions where predictors disagree most")
add_bar_labels(ax, fontsize=9)
add_note(fig, "Higher values mean predictor scores are more spread out after z-score scaling.")
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(OUTPUT_DIR / "top_contested_regions.png", dpi=200, bbox_inches="tight")
plt.show()

contested_top[contested_display_columns]


### Contested Region Case Studies

The table below turns the contested-region ranking into inspectable case studies. For each top region, it adds the human protein name/gene when available and reports which predictor is highest and lowest at the single most-disagreed residue in the window. Large `pdbflex` outlier values are especially useful for checking whether disagreement is driven by flexibility-like signal rather than canonical disorder-like signal.

In [ ]:
def parse_human_fasta_headers(path):
    rows = []
    with path.open() as handle:
        for line in handle:
            if not line.startswith(">"):
                continue
            header = line[1:].strip()
            first = header.split()[0]
            parts = first.split("|")
            accession = parts[1] if len(parts) >= 2 else first
            gene_match = __import__("re").search(r"\bGN=([^\s]+)", header)
            protein_name = header.split(" OS=")[0]
            if " " in protein_name:
                protein_name = protein_name.split(" ", 1)[1]
            rows.append(
                {
                    "protein_id": accession,
                    "human_gene": gene_match.group(1) if gene_match else "",
                    "human_protein_name": protein_name,
                    "human_header": header,
                }
            )
    return pd.DataFrame(rows).drop_duplicates("protein_id")

human_header_df = parse_human_fasta_headers(ROOT / "HumanProteome" / "human_preteome.fasta")
score_columns = [column for column in contested_regions.columns if column.endswith("_scaled_score_at_max")]

def predictor_name_from_score_column(column):
    return column.replace("_scaled_score_at_max", "")

case_studies = contested_regions.sort_values(["mean_std", "max_std"], ascending=False).head(20).copy()
case_studies["region"] = (
    case_studies["protein_id"].astype(str)
    + ":"
    + case_studies["start_residue"].astype(str)
    + "-"
    + case_studies["end_residue"].astype(str)
)
case_studies["highest_predictor_at_max"] = case_studies[score_columns].idxmax(axis=1).map(predictor_name_from_score_column)
case_studies["lowest_predictor_at_max"] = case_studies[score_columns].idxmin(axis=1).map(predictor_name_from_score_column)
case_studies["highest_scaled_score_at_max"] = case_studies[score_columns].max(axis=1)
case_studies["lowest_scaled_score_at_max"] = case_studies[score_columns].min(axis=1)
case_studies["pdbflex_minus_median_at_max"] = (
    case_studies["pdbflex_scaled_score_at_max"] - case_studies[score_columns].median(axis=1)
)
case_studies = case_studies.merge(human_header_df, on="protein_id", how="left")

case_study_columns = [
    "region",
    "human_gene",
    "human_protein_name",
    "max_disagreement_residue",
    "mean_std",
    "max_std",
    "highest_predictor_at_max",
    "highest_scaled_score_at_max",
    "lowest_predictor_at_max",
    "lowest_scaled_score_at_max",
    "pdbflex_scaled_score_at_max",
    "pdbflex_minus_median_at_max",
]
case_studies[case_study_columns]

In [ ]:
plot_cases = case_studies.sort_values("pdbflex_minus_median_at_max", ascending=False).head(15).copy()
plot_cases["highest_predictor_label"] = plot_cases["highest_predictor_at_max"].map(display_name)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(
    data=plot_cases,
    x="pdbflex_minus_median_at_max",
    y="region",
    hue="highest_predictor_label",
    dodge=False,
    ax=ax,
)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("PDBFlex score minus the median predictor score at the most-disagreed residue")
ax.set_ylabel("Contested region")
ax.set_title("Is PDBFlex the high-scoring outlier in contested regions?")
ax.legend(title="Highest predictor", loc="lower right", frameon=True)
add_note(fig, "Positive values mean PDBFlex is above the median predictor at the most-disagreed residue.")
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(OUTPUT_DIR / "contested_regions_pdbflex_outlier_profile.png", dpi=200, bbox_inches="tight")
plt.show()
